# Module 2 Homework: Vector Search

In [ ]:
import sys
sys.path.insert(0, '.')  # embedder.py lives here

import numpy as np
from embedder import Embedder

embedder = Embedder()

## Q1. Embed the query, check v[0]

In [ ]:
query_q1 = "How does approximate nearest neighbor search work?"
v = embedder.encode(query_q1)

print(f"Vector length : {len(v)}")
print(f"Q1 answer v[0]: {v[0]:.4f}")

## Load documents (72 lesson pages, pinned to commit 8c1834d)

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

## Q2. Cosine similarity with 07-sqlitesearch-vector.md

In [ ]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = next(d for d in documents if d["filename"] == target)

doc_vec = embedder.encode(doc["content"])
similarity = float(np.dot(v, doc_vec))  # vectors are normalized → dot == cosine sim

print(f"Q2 cosine similarity: {similarity:.4f}")

## Q3. Chunk, embed, score against Q1 query by hand

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

In [ ]:
# Embed all chunks — returns (n_chunks, 384) numpy array
contents = [c["content"] for c in chunks]
X = embedder.encode_batch(contents)
print(f"Embedding matrix: {X.shape}")

In [ ]:
scores = X.dot(v)
best_idx = int(np.argmax(scores))

print(f"Best score   : {scores[best_idx]:.4f}")
print(f"Q3 filename  : {chunks[best_idx]['filename']}")

## Q4. VectorSearch with minsearch — new query

In [ ]:
from minsearch import VectorSearch

# VectorSearch.fit(vectors_2d_array, payload_list)
vec_index = VectorSearch(keyword_fields=["filename", "start"])
vec_index.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vec = embedder.encode(q4_query)

vec_results_q4 = vec_index.search(q4_vec, num_results=5)

print("Top 5 vector results:")
for i, r in enumerate(vec_results_q4):
    print(f"  {i+1}. {r['filename']}")
print(f"\nQ4 answer: {vec_results_q4[0]['filename']}")

## Q5. Vector vs Text search — find what's only in vector results

In [ ]:
from minsearch import Index

txt_index = Index(text_fields=["content"], keyword_fields=["filename", "start"])
txt_index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vec = embedder.encode(q5_query)

vec_results_q5 = vec_index.search(q5_vec, num_results=5)
txt_results_q5 = txt_index.search(q5_query, num_results=5)

vec_files = {r["filename"] for r in vec_results_q5}
txt_files = {r["filename"] for r in txt_results_q5}

print("Vector top-5:")
for r in vec_results_q5:
    print(f"  {r['filename']}")

print("\nText top-5:")
for r in txt_results_q5:
    print(f"  {r['filename']}")

only_vector = vec_files - txt_files
print(f"\nQ5 answer (in vector but NOT text): {only_vector}")

## Q6. Hybrid search with RRF

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6_query = "How do I give the model access to tools?"
q6_vec = embedder.encode(q6_query)

vec_results_q6 = vec_index.search(q6_vec, num_results=5)
txt_results_q6 = txt_index.search(q6_query, num_results=5)

print("Vector top-5:")
for r in vec_results_q6:
    print(f"  {r['filename']}")

print("\nText top-5:")
for r in txt_results_q6:
    print(f"  {r['filename']}")

results_q6 = rrf([vec_results_q6, txt_results_q6])

print("\nRRF fused top-5:")
for i, r in enumerate(results_q6):
    print(f"  {i+1}. {r['filename']}")
print(f"\nQ6 answer: {results_q6[0]['filename']}")

## All answers

In [ ]:
print("=" * 55)
print("ANSWERS")
print("=" * 55)
print(f"Q1  v[0]                      = {v[0]:.4f}")
print(f"Q2  cosine sim (doc vs query)  = {similarity:.4f}")
print(f"Q3  top chunk filename         = {chunks[best_idx]['filename']}")
print(f"Q4  first vector result        = {vec_results_q4[0]['filename']}")
print(f"Q5  only in vector (not text)  = {only_vector}")
print(f"Q6  first after RRF            = {results_q6[0]['filename']}")